# Build 03-03 · Retrain + score per axis — inside the version env

**Kernel: that version's own env** (`fttl-v2` / `fttl-v3` — the 00_SHAP kernels). Run this
notebook **twice, once per kernel**; it names no version — it detects which env it is running
in and refuses any other (env-v1 is out of scope: v1 is not retrained).

Retraining and scoring live in ONE session on purpose: the moment a model is fitted, the same
kernel scores the splits — no second env round-trip. Both steps launch the CLI workers with
**this kernel's own interpreter**, so notebook and CLI cannot drift:

```
mitigation/<v>_corrected_<split>_<tag>.parquet   (from 03_02)
   ──▶  §2 retrain.py (this kernel)  ──▶  models/real/mitigated/<v>_<split>_<tag>.pkl (+ _meta.json)
   ──▶  §4 predict.py (this kernel)  ──▶  reeval/<v>_mitigated_scores_<score-split>_<tag>.parquet
   ──▶  03_04 (score-file diagnostics)  ──▶  03_05 (evaluation) — both analysis env
```

`retrain.py` clones the baseline estimator's hyperparameters (the production pickle is only
read) and fits fresh on the corrected labels/weights: the target is the only thing that
changes (re-evaluation invariant). Note `sample_weight` and the cloned `scale_pos_weight`
**multiply** in XGBoost; every axis carries the same factor, so the comparison stays fair.
`predict.py` takes the feature columns from the fitted estimator in trained order, so scoring
cannot disagree with the fit.

In [ ]:
# §0 — setup: detect which version env this kernel is, verify the xgboost pin
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config

exe = Path(sys.executable).resolve()
VERSION = None
for v in ("v2", "v3"):
    try:
        if Path(config.python_bin(v)).resolve() == exe:
            VERSION = v
    except FileNotFoundError:
        pass
if VERSION is None:
    raise SystemExit(
        "kernel " + str(exe) + " is neither env-v2 nor env-v3. Run this notebook on a version "
        "kernel (fttl-v2 / fttl-v3 - see notebook/real/README.md for the ipykernel setup); "
        "v1 is out of scope."
    )

import xgboost
pin = config.xgboost_pin(VERSION)
assert xgboost.__version__ == pin, (
    "xgboost " + xgboost.__version__ + " != pinned " + str(pin) + " for " + VERSION
    + " - wrong kernel/env, a mismatched load produces figures, not errors"
)
print("VERSION =", VERSION, "| xgboost", xgboost.__version__)
print("kernel  =", exe)

In [ ]:
# §1 — RUN_SPEC + discover 03_02's corrected files for THIS version
SPLIT = "train"      # the split 03_02 corrected (its RUN_SPEC SPLIT)
ID_COL = "claim_id"
SCORE_SPLITS = list(dict.fromkeys([SPLIT, config.OOT_SPLIT[VERSION]]))
                     # OOT never picked by name — v1/v2 invert "test"

MIT_DIR = ROOT / "src" / "data" / "real" / "mitigation"
MODEL_DIR = ROOT / "src" / "models" / "real" / "mitigated"
REEVAL_DIR = ROOT / "src" / "data" / "real" / "reeval"
FEATURES_PATH = config.split_path("processed_inputs", VERSION, SPLIT)
BASELINE_PKL = config.path("model", VERSION)     # hyperparameter donor — read, never written
REGISTRY = config.registry_path(VERSION)
RETRAIN_PY = ROOT / "src" / "training" / "retrain.py"
PREDICT_PY = ROOT / "src" / "scoring" / "predict.py"

prefix = VERSION + "_corrected_" + SPLIT + "_"
corrected = sorted(p for p in MIT_DIR.glob(prefix + "*.parquet"))
TAGS = [p.stem[len(prefix):] for p in corrected]
assert TAGS, "no " + prefix + "*.parquet under " + str(MIT_DIR) + " - run 03_02 first"

print("baseline :", BASELINE_PKL)
print("features :", FEATURES_PATH)
print("axes     :", TAGS)
print("score on :", SCORE_SPLITS)

In [ ]:
# §2 — retrain one model per axis (worker = retrain.py, this kernel's interpreter)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_paths = {}
for tag, labels_path in zip(TAGS, corrected):
    out_pkl = MODEL_DIR / (VERSION + "_" + SPLIT + "_" + tag + ".pkl")
    cmd = [str(exe), str(RETRAIN_PY),
           "--baseline", str(BASELINE_PKL), "--features", str(FEATURES_PATH),
           "--labels", str(labels_path), "--version", VERSION,
           "--out-model", str(out_pkl), "--id-col", ID_COL,
           "--features-json", str(REGISTRY)]
    print(">>", " ".join(cmd))
    subprocess.run(cmd, cwd=str(ROOT), check=True)
    model_paths[tag] = out_pkl

In [ ]:
# §3 — what was fitted: the retrain meta sidecars, side by side
rows = []
for tag, pkl in model_paths.items():
    meta = json.loads(Path(str(pkl)[: -len(".pkl")] + "_meta.json").read_text(encoding="utf-8"))
    rows.append({"run": tag, "n_rows": meta["n_rows"], "n_features": meta["n_features"],
                 "feature_selection": meta["feature_selection"], "weighted": meta["weighted"],
                 "weight_mean": meta["weight_mean"], "weight_max": meta["weight_max"],
                 "label_pos_rate": meta["label_pos_rate"], "estimator": meta["estimator"]})
display(pd.DataFrame(rows).set_index("run"))

In [ ]:
# §4 — score every (axis, split) pair (worker = predict.py, this kernel's interpreter)
REEVAL_DIR.mkdir(parents=True, exist_ok=True)
score_paths = {}
for tag, pkl in model_paths.items():
    for sp in SCORE_SPLITS:
        out_sc = REEVAL_DIR / (VERSION + "_mitigated_scores_" + sp + "_" + tag + ".parquet")
        cmd = [str(exe), str(PREDICT_PY),
               "--model", str(pkl),
               "--features", str(config.split_path("processed_inputs", VERSION, sp)),
               "--version", VERSION, "--out", str(out_sc), "--id-col", ID_COL,
               "--features-json", str(REGISTRY)]
        print(">>", " ".join(cmd))
        subprocess.run(cmd, cwd=str(ROOT), check=True)
        score_paths[(tag, sp)] = out_sc
print("wrote", len(score_paths), "score files -> ", REEVAL_DIR)

## Notes

- Naming: the split in the **score file** is the split *scored*; the split in the **pkl** is
  the split *trained on*. Train-split scores feed 03_05's provisional threshold tuning; the
  OOT split feeds the evaluation.
- Baseline scores are NOT produced here — they are the `scores` kind
  (`src/data/real/detection/<v>_scores_<split>.parquet`, from `src/scoring/score_all.py`);
  03_04/03_05 pick them up from there when present.
- Next: **03_04_prediction.ipynb** (score-file diagnostics) and **03_05_reevaluation.ipynb**
  (operating points) — both on the analysis `.venv` kernel.